# forja-pretrain on Kaggle (or Colab)

Pretrains the `forja-pretrain` Transformer (~10.6M non-embedding parameters, Qwen2.5-Coder tokenizer) on a free T4 GPU.

**Before running (Kaggle):**
1. *Settings → Accelerator*: **GPU T4 x1** (just one; the code does not use both).
2. *Settings → Internet*: **on** (requires a verified phone number on the account).
3. Optional, to publish to the Hugging Face Hub: *Add-ons → Secrets* → `HF_TOKEN` (a token with write permission).
4. For long training, use **Save Version → Save & Run All (Commit)**: it runs in the background for up to 12 h and saves `/kaggle/working` as output. An interactive session dies if left idle.

**If the session ends before training finishes:** add the previous version's *output* of this notebook as an input (*Add Input → Your Work*) and run again. Cell 4 finds `last.pt` and training continues from the same step, with a result identical to continuous training (tested in `tests/test_train.py`).

**On Colab:** *Runtime → Change runtime type → T4 GPU*. The notebook mounts Google Drive and saves everything there, so resuming is automatic.

In [ ]:
# 1. Configuração: edite aqui
REPO_URL = "https://github.com/josuekuhim/forja-pretrain"  # publique o repo antes (ou use um Kaggle Dataset com o código)
REPO_REF = "main"
CONFIG = "configs/kaggle-t4.yaml"
DATA_CONFIG = "configs/data.yaml"
TOTAL_TOKENS = 300_000_000   # orçamento do corpus; 8000 passos x 32k tokens = ~262M
HF_REPO_ID = None            # ex.: "josuekuhim/forja-pretrain-d384-l6"; None = não publica
EXTRA_OVERRIDES = []         # ex.: ["model.n_kv_heads=2", "train.out_dir=runs/gqa2"] para ablações

In [ ]:
# 2. Ambiente, GPU e diretórios
import os, sys, subprocess, json, shutil, glob
from pathlib import Path

IN_KAGGLE = Path("/kaggle/working").exists()
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/forja-pretrain")
elif IN_KAGGLE:
    WORK = Path("/kaggle/working")
else:
    WORK = Path.cwd() / "forja-work"
WORK.mkdir(parents=True, exist_ok=True)
DATA_DIR, RUNS_DIR = WORK / "data", WORK / "runs"
print("WORK =", WORK)

import torch
assert torch.cuda.is_available(), "Sem GPU: ligue o acelerador T4 nas configurações do notebook."
print(torch.__version__, torch.cuda.get_device_name(0),
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 3. Código do forja-pretrain (o torch com CUDA já vem instalado no Kaggle/Colab)
REPO = Path("/tmp/forja-pretrain") if (IN_KAGGLE or IN_COLAB) else WORK / "forja-pretrain"
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[data]"], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 4. Retomada: traz o last.pt de uma versão anterior (input do Kaggle) se existir
import yaml
cfg = yaml.safe_load(open(CONFIG, encoding="utf-8"))
run_name = Path(cfg["train"]["out_dir"]).name
for o in EXTRA_OVERRIDES:
    if o.startswith("train.out_dir="):
        run_name = Path(o.split("=", 1)[1]).name
RUN_DIR = RUNS_DIR / run_name

if not (RUN_DIR / "last.pt").exists():
    anteriores = sorted(glob.glob(f"/kaggle/input/**/runs/{run_name}/last.pt", recursive=True))
    if anteriores:
        src = Path(anteriores[-1]).parent
        shutil.copytree(src, RUN_DIR, dirs_exist_ok=True)
        print("retomando a partir de", src)
    else:
        print("treino novo:", RUN_DIR)
else:
    print("last.pt já está em", RUN_DIR, "-> o treino retoma de lá")

In [ ]:
# 5. Dados: baixa (streaming), tokeniza e grava data/{train,val}.bin. Pula se já existir.
#    Com uma versão anterior como input, reaproveita os .bin dela em vez de baixar de novo.
if not (DATA_DIR / "train.bin").exists():
    prev = sorted(glob.glob("/kaggle/input/**/data/train.bin", recursive=True))
    if prev:
        shutil.copytree(Path(prev[-1]).parent, DATA_DIR, dirs_exist_ok=True)
        print("dados reaproveitados de", Path(prev[-1]).parent)
    else:
        dcfg = yaml.safe_load(open(DATA_CONFIG, encoding="utf-8"))
        dcfg["out_dir"] = str(DATA_DIR)
        dcfg["total_tokens"] = TOTAL_TOKENS
        tmp_cfg = WORK / "data_config.yaml"
        tmp_cfg.write_text(yaml.safe_dump(dcfg, allow_unicode=True), encoding="utf-8")
        subprocess.run([sys.executable, "scripts/prepare_data.py", str(tmp_cfg)], check=True)
print(json.dumps(json.load(open(DATA_DIR / "manifest.json", encoding="utf-8")), indent=2, ensure_ascii=False)[:2000])

In [ ]:
# 6. Treino. Imprime tok/s nos primeiros passos: use para estimar o tempo total.
#    Se bater train.max_hours, salva o last.pt e para; rode a próxima versão para continuar.
overrides = [
    f"data.train_bin={DATA_DIR / 'train.bin'}",
    f"data.val_bin={DATA_DIR / 'val.bin'}",
    f"train.out_dir={RUN_DIR}",
    *EXTRA_OVERRIDES,
]
proc = subprocess.Popen(["forja-train", CONFIG, *overrides], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
assert proc.wait() == 0, "treino falhou; veja o log acima"
summary = json.load(open(RUN_DIR / "summary.json", encoding="utf-8"))
summary

In [ ]:
# 7. Gráficos de loss e perplexidade
from IPython.display import Image, display
png = subprocess.run([sys.executable, "scripts/plot_history.py", str(RUN_DIR), "--out", str(RUN_DIR / "loss.png")],
                     check=True, capture_output=True, text=True).stdout.strip()
display(Image(png))

In [ ]:
# 8. Exporta no formato do contrato (safetensors + config.json + model card) e publica, se configurado
EXPORT_DIR = WORK / "export" / run_name
cmd = [sys.executable, "scripts/export_hub.py", str(RUN_DIR), str(EXPORT_DIR)]
if HF_REPO_ID and summary.get("concluido"):
    if IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    elif IN_COLAB:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    cmd += ["--push", HF_REPO_ID]
elif HF_REPO_ID:
    print("treino ainda não concluído: exporto localmente, mas só publico no Hub quando terminar")
subprocess.run(cmd, check=True)
print(sorted(p.name for p in EXPORT_DIR.iterdir()))

## After training

- Copy `summary.json`, `loss.png` and the Hub link into the `forja-pretrain` README (*Result* section).
- Ablations (one variable at a time): run again with `EXTRA_OVERRIDES = ["model.n_kv_heads=2", "train.out_dir=runs/gqa2"]`, and compare with `scripts/plot_history.py runs/a runs/b`.
- The system's next step: measure this model's acceptance rate as a draft for Qwen2.5-Coder (anticipates `forja-serve`).